# MAS

In [1]:
from src.board.core import Board

board = Board()
tools = board.tools
tools

[StructuredTool(name='add_tasks', description='Добавляет перечень задач на доску задач\nВозвращает список ID задач в порядке добавления.', args_schema=<class 'langchain_core.utils.pydantic.add_tasks'>, func=<bound method Board.add_tasks of Board(iteration=0, tasks=[], results=[])>),
 StructuredTool(name='update_status', description='Обновляет текущий статус задачи.\nВозвращает None, если задача с таким ID не найдена.\nВозвращает текущее состояние задачи.\nВНИМАНИЕ: сверяйте желаемое и текущее состояние задачи.', args_schema=<class 'langchain_core.utils.pydantic.update_status'>, func=<bound method Board.update_status of Board(iteration=0, tasks=[], results=[])>),
 StructuredTool(name='get_results', description='Возвращает результаты, относящиеся к задаче. \nВозвращает None, если задача с таким ID не найдена.', args_schema=<class 'langchain_core.utils.pydantic.get_results'>, func=<bound method Board.get_results of Board(iteration=0, tasks=[], results=[])>),
 StructuredTool(name='add_resu

In [2]:
from langgraph.graph import MessagesState

class State(MessagesState):
    last_messages_len : int
    manager_response : str

In [3]:
from src import Agent
from langchain.messages import AIMessage


manager_tools = [tool for tool in tools if tool.name not in ['add_result']]

manager_prompt = """
Ты менеджер. 

ТВОИ ОБЯЗАННОСТИ:
- Декомпозиция запроса пользователя и создание задач для агентов.
- Отслеживание прогресса по задачам.
- Суммаризация результатов и формирование ответа пользователю.

ТЫ ТАКЖЕ МОЖЕШЬ:
- Помечать задачи как DONE или CANCELLED.
- Менять приоритет задач. 

ВАЖНО:
- Работа заканчивается тогда, когда все задачи на доске либо CANCELLED, либо DONE.
- Если ты готов отдать ответ пользователю, убедись, что все задачи имеют необходимый статус для завершения.
- Агенты ничего не знают про запрос пользователя. При постановке задачи убедись, что информации достаточно.

ТЫ НЕ ДОЛЖЕН отвечать на основе своих знаний. Отвечай только на основе результатов по задачам.

ЗАПРОС ПОЛЬЗОВАТЕЛЯ:
{query}
"""

manager_agent = Agent(tools=manager_tools, system_prompt=manager_prompt.format(query="""
Анализ потенциала г. Гатчина (Ленинградская область)
"""))

def manager_node(state : State):
    board.update_iteration()
    
    messages = state['messages']
    last_messages_len = state.get('last_messages_len',0)
    new_messages = messages[last_messages_len:]

    result = manager_agent.run(new_messages)

    return {
        'manager_response': result,
        'last_messages_len': len(messages)  # обновляем индекс
    }

In [4]:
def loop_node(state : State):
    return {}

In [5]:
agent_prompt = """
Ты {role}. Твои обязанности:
- Проверка пула новых задач.
- Выполнение задач, относящихся к твоей специализации.

ПРАВИЛА РАБОТЫ:
- Если можешь, выполняй задачу без лишних слов.
- Если чего-то не хватает, напиши.
- Если не относится к тебе - ничего не делай.
- Ты не можешь помечать задачи как CANCELLED или DONE.
"""

agent_tools = [tool for tool in tools if tool.name not in ['update_priority', 'add_tasks']]

agents = {role:Agent(
    tools=agent_tools, 
    system_prompt=agent_prompt.format(role=role)    
) for role in [
    'специалист по стратегическому планированию',
    'специалист по территориальному планированию',
    'специалист по урбанистике',
]}

def create_node(agent):

    def agent_node(state : State):
        message = AIMessage(state['manager_response'])
        result = agent.run([message])
        return {'messages': [AIMessage(result)]}     
    
    return agent_node    

agents_nodes = {role: create_node(agent) for role,agent in agents.items()}

In [6]:
from langgraph.graph import StateGraph, START, END
from src.board.models import Status

graph = StateGraph(State)

graph.add_node('manager', manager_node)
graph.add_node('loop', loop_node)

def loop_condition(state : State):
    tasks = board.get_tasks()
    finish = all([t.status in [Status.CANCELLED, Status.DONE] for t in tasks])
    if finish:
        return END
    return 'loop'
        
graph.add_edge(START, 'manager')
graph.add_conditional_edges('manager', loop_condition, [END, 'loop'])

for role,node in agents_nodes.items():
    graph.add_node(role, node)
    graph.add_edge('loop', role)
    graph.add_edge(role, 'manager')


app = graph.compile(debug=True)

In [7]:
app.invoke({'messages': []})

[values] {'messages': []}
[updates] {'manager': {'manager_response': '', 'last_messages_len': 0}}
[values] {'messages': [], 'last_messages_len': 0, 'manager_response': ''}
[updates] {'loop': None}
[updates] {'специалист по территориальному планированию': {'messages': [AIMessage(content='Для выполнения указанных задач требуется доступ к актуальной статистической, экономической и иной профильной информации о городе Гатчина (данные Росстата, региональные аналитические отчёты, открытые реестры предприятий, туристические статистики, инфраструктурные карты и т.п.). Без доступа к этим источникам я не могу собрать требуемые данные и сформировать результаты. Пожалуйста, предоставьте доступ к необходимым источникам или уточните, какие именно данные уже имеются в наличии.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}}
[updates] {'специалист по урбанистике': {'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={}, tool_calls=[], invali

APIConnectionError: Connection error.

In [8]:
board.model_dump()

{'iteration': 12,
 'tasks': [{'name': 'Demographics',
   'description': 'Collect latest demographic statistics for Gatchina (population, age structure, growth trends).',
   'priority': <Priority.CRITICAL: 5>,
   'id': '6976400c5e5e442c9acc15cf88c7ecc6',
   'status': <Status.TODO: 'todo'>,
   'created_at': 1},
  {'name': 'EconomicData',
   'description': 'Gather economic data: GDP, main industries, employment rates, average income for Gatchina.',
   'priority': <Priority.CRITICAL: 5>,
   'id': '5c6fc5e4291f4aa5b509a0f97e06db9f',
   'status': <Status.TODO: 'todo'>,
   'created_at': 1},
  {'name': 'IndustrialEnterprises',
   'description': 'Identify key industrial enterprises and sectors operating in Gatchina.',
   'priority': <Priority.CRITICAL: 5>,
   'id': '2255014ec88d4365880caaf3a89b4cb2',
   'status': <Status.TODO: 'todo'>,
   'created_at': 1},
  {'name': 'TourismPotential',
   'description': 'Analyze tourism potential: historical sites, museums, parks, visitor numbers in Gatchina.'